# Mytho — Recurrent-Depth Transformer
Pretrain Mytho models (10M to 7B) on FineWeb-Edu using **Kaggle 2x T4 GPUs**.

**Setup:** `Settings -> Accelerator -> GPU T4 x2`

With FSDP across 2 GPUs, the model is sharded — roughly 2x the capacity vs single-GPU Colab.

## 1. Environment Setup

In [ ]:
import torch
n_gpus = torch.cuda.device_count()
assert n_gpus > 0, "No GPU found - enable GPU T4 x2 in Settings"
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i}: {name}  |  {vram:.1f} GB")
print(f"\nTotal GPUs: {n_gpus}  |  Total VRAM: {sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpus)) / 1e9:.1f} GB")

In [ ]:
!pip install -q tiktoken>=0.5 datasets>=2.16 tqdm>=4.65 matplotlib bitsandbytes safetensors onnx
print("Dependencies installed")

## 2. Clone Repository

In [ ]:
import os
os.chdir("/kaggle/working")
REPO = "Mytho"
if not os.path.exists(REPO):
    !git clone https://github.com/dev-760/Mytho.git $REPO
else:
    !cd $REPO && git pull
os.chdir(REPO)
print(f"Working dir: {os.getcwd()}")

## 3. Choose Model Size

With **2x T4 + FSDP**, model weights are sharded across GPUs — larger configs fit.

| Config | Params | VRAM (1x T4) | VRAM (2x T4 FSDP) | Recommended |
|--------|--------|-------------|-------------------|-------------|
| **10M** | ~10M | <1 GB | <1 GB | ✅ Smoke test |
| **50M** | ~50M | ~1-2 GB | ~1 GB | ✅ Quick exp |
| **100M** | ~100M | ~2-3 GB | ~1-2 GB | ✅ Fast iter |
| **150M** | ~150M | ~3-4 GB | ~2 GB | ✅ |
| **500M** | ~500M | ~5-6 GB | ~3-4 GB | ✅ Sweet spot |
| **1B** | ~1.05B | ~8-10 GB | ~5-6 GB | ✅ Comfortable |
| **3B** | ~3.2B | ~13-15 GB | ~8-10 GB | ✅ Fits well |
| **7B** | ~7B | ❌ | ~14-15 GB | ⚠️ Tight, needs offload |

In [ ]:
#@title Select Model Size
MODEL_SIZE = "500M"  # @param ["10M", "50M", "100M", "150M", "500M", "1B", "3B", "7B"]

from model_configs import MODEL_CONFIGS
MODEL_ARGS = MODEL_CONFIGS[MODEL_SIZE]

print(f"Selected: Mytho-{MODEL_SIZE}")
for k, v in MODEL_ARGS.items():
    print(f"  {k}: {v}")

## 4. Verify Model Builds

In [ ]:
import torch
from mytho_model import MythoConfig, MythoModel
from data import VOCAB_SIZE

config = MythoConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_ARGS["d_model"],
    n_heads=MODEL_ARGS["n_heads"],
    d_head=MODEL_ARGS["d_head"],
    d_latent_kv=MODEL_ARGS["d_latent_kv"],
    d_rope=MODEL_ARGS["d_rope"],
    n_experts=MODEL_ARGS["n_experts"],
    n_active_experts=MODEL_ARGS["n_active_experts"],
    d_expert_ff=MODEL_ARGS["d_expert_ff"],
    max_depth=MODEL_ARGS["max_depth"],
    max_seq_len=MODEL_ARGS["seq_len"],
    n_unique_blocks=MODEL_ARGS["n_unique_blocks"],
    dropout=0.0,
)

model = MythoModel(config).cuda()
n_params = model.num_parameters()
print(f"Mytho-{MODEL_SIZE}: {n_params:,} params ({n_params/1e6:.1f}M)")

ids = torch.randint(1, config.vocab_size, (1, 32), device="cuda")
with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
    out = model(ids, labels=ids)
print(f"Forward pass OK — Loss: {out['loss'].item():.4f}, Depth: {out['mean_depth'].item():.1f}")

del model, ids, out
torch.cuda.empty_cache()

## 5. Checkpoint Directory

In [ ]:
CKPT_DIR = f"/kaggle/working/checkpoints_mytho_{MODEL_SIZE}"
print(f"Checkpoints: {CKPT_DIR}")

## 6. Pretrain (2x T4 with FSDP)

Uses `torchrun` to launch FSDP across both GPUs. The model is sharded automatically.

- **FP16** mixed-precision (T4 does not support BF16)
- **Gradient clipping** enabled via `--max_grad_norm`
- **Faster checkpoints** via `--state_dict_type sharded`
- **CPU offload** available for 7B via `--cpu_offload`
- **Activation checkpointing** enabled by default
- Saves `.pt` at each checkpoint (safetensors only for full state)

In [ ]:
import torch
N_GPUS = torch.cuda.device_count()
assert N_GPUS > 0, "No GPU found - enable GPU T4 x2 in Settings"

extra_flags = "--cpu_offload" if MODEL_SIZE == "7B" else ""

cmd = f"torchrun --nproc_per_node={N_GPUS} --master_port=29500 pretrain.py " \
      f"--model_size {MODEL_SIZE} --dtype fp16 --max_steps 2000 --warmup_steps 100 " \
      f"--lr 3e-4 --max_grad_norm 1.0 --save_every 1000 --state_dict_type sharded " \
      f"--log_every 10 --ckpt_dir {CKPT_DIR} {extra_flags}"

print(f"Training Mytho-{MODEL_SIZE} on {N_GPUS}x T4")
print(cmd)
!{cmd}

### Alternative: Single-GPU (pretrain_t4.py)
If you prefer a lighter single-GPU loop (works on P100) or FSDP gives issues:

In [ ]:
# Uncomment to use single-GPU T4-optimised script instead:
# !python pretrain_t4.py --model_size {MODEL_SIZE} \
#     --grad_checkpoint --optim_8bit \
#     --max_steps 2000 --warmup_steps 100 --lr 3e-4 --max_grad_norm 1.0 \
#     --log_every 10 --save_every 1000 \
#     --ckpt_dir {CKPT_DIR}

## 7. Training Curves

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

log_path = Path(CKPT_DIR) / "train_log.jsonl"
if log_path.exists():
    records = [json.loads(l) for l in log_path.read_text().strip().split("\n") if l.strip()]
    steps  = [r["step"] for r in records]
    losses = [r["loss"] for r in records]
    ce     = [r["ce_loss"] for r in records]
    depths = [r["mean_depth"] for r in records]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Mytho-{MODEL_SIZE} Training (Kaggle 2xT4)", fontsize=14, fontweight="bold")
    axes[0].plot(steps, losses); axes[0].set_title("Total Loss"); axes[0].grid(alpha=0.3)
    axes[1].plot(steps, ce);     axes[1].set_title("CE Loss");    axes[1].grid(alpha=0.3)
    axes[2].plot(steps, depths); axes[2].set_title("Mean Depth"); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Final loss: {losses[-1]:.4f}")
else:
    print("No training log found. Run training first.")

## 8. Generate Text

In [ ]:
import torch, glob
from mytho_model import MythoModel
from data import tokenise, decode

ckpts = sorted(glob.glob(f"{CKPT_DIR}/step_*.pt"))
if ckpts:
    ckpt = torch.load(ckpts[-1], map_location="cuda:0", weights_only=False)
    model = MythoModel(ckpt["config"]).cuda()
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Loaded step {ckpt['step']}  |  {model.num_parameters():,} params")

    for prompt in ["The history of artificial intelligence", "In mathematics, a prime number is"]:
        input_ids = torch.tensor([tokenise(prompt)], device="cuda")
        with torch.no_grad():
            out = model.generate(input_ids, max_new_tokens=80, temperature=0.8, top_k=50)
        print(f"\nPrompt: {prompt}")
        print(f"Output: {decode(out[0].tolist())}")

    del model
    torch.cuda.empty_cache()
else:
    print("No checkpoints found. Run training first.")

## 9. Chat with Mytho
Type a message, get a response. Type `quit` to stop.

In [ ]:
import torch, glob
from mytho_model import MythoModel
from data import tokenise, decode

ckpts = sorted(glob.glob(f"{CKPT_DIR}/step_*.pt"))
assert ckpts, "No checkpoints found. Run training first."

ckpt = torch.load(ckpts[-1], map_location="cuda:0", weights_only=False)
model = MythoModel(ckpt["config"]).cuda()
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Mytho-{MODEL_SIZE} loaded (step {ckpt['step']}, {model.num_parameters():,} params)")
print("Type your message below. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        print("Chat ended.")
        break
    if not user_input.strip():
        continue
    input_ids = torch.tensor([tokenise(user_input)], device="cuda")
    with torch.no_grad():
        out_ids = model.generate(input_ids, max_new_tokens=150, temperature=0.8, top_k=50, top_p=0.9)
    print(f"Mytho: {decode(out_ids[0, input_ids.shape[1]:].tolist())}\n")

del model
torch.cuda.empty_cache()

## 10. Export to Safetensors
Portable weight format. Already saved alongside `.pt` during training, or export manually:

In [ ]:
import glob
EXPORT_DIR = f"{CKPT_DIR}/exports"
ckpts = sorted(glob.glob(f"{CKPT_DIR}/step_*.pt"))
if ckpts:
    latest = ckpts[-1]
    print(f"Exporting {latest} to safetensors...")
    !python export.py --checkpoint {latest} --format safetensors --output_dir {EXPORT_DIR}
else:
    print("No checkpoints found. Run training first.")

## 11. Export to ONNX
Inference-only graph for cross-platform deployment. ACT runs at fixed `max_depth`.

In [ ]:
import glob
EXPORT_DIR = f"{CKPT_DIR}/exports"
ckpts = sorted(glob.glob(f"{CKPT_DIR}/step_*.pt"))
if ckpts:
    latest = ckpts[-1]
    print(f"Exporting {latest} to ONNX...")
    !python export.py --checkpoint {latest} --format onnx --output_dir {EXPORT_DIR}
else:
    print("No checkpoints found. Run training first.")

## 12. Save to Kaggle Output
All files in `/kaggle/working/` are available in the notebook output tab.

In [ ]:
import glob, os

print(f"=== Checkpoints ({CKPT_DIR}) ===")
for ext in ['*.pt', '*.safetensors']:
    for f in sorted(glob.glob(f"{CKPT_DIR}/{ext}")):
        print(f"  {os.path.basename(f):40s}  {os.path.getsize(f)/1e6:.1f} MB")

export_dir = f"{CKPT_DIR}/exports"
if os.path.exists(export_dir):
    print(f"\n=== Exports ({export_dir}) ===")
    for f in sorted(glob.glob(f"{export_dir}/*")):
        print(f"  {os.path.basename(f):40s}  {os.path.getsize(f)/1e6:.1f} MB")

print("\nGo to: Output tab → Download or create a Kaggle Dataset.")